# Training model with distributed learning pipeline


## Verification of dependencies and their versions

In [1]:
!pip list | grep -i kubeflow

kubeflow                  0.4.0
kubeflow_katib_api        0.19.0
kubeflow_trainer_api      2.2.1


## PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes. `TrainerClient()` verifies that you have required access to the Kubernetes cluster. Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [2]:
import kubeflow
from kubeflow.trainer import CustomTrainer, TrainerClient
config = kubeflow.trainer.KubernetesBackendConfig()

client = TrainerClient()
trainer = TrainerClient(backend_config=config)

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob. Additionally, it might show available accelerator type and number of available resources.

In [3]:
for runtime in trainer.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.2.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='jax-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='jax', image='nvcr.io/nvidia/jax:25.10-py3', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None)
Runtime(name='med-3d-configurable', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/med_3d_runner:v0.0.2', num_nodes=1, device='gpu', device_count='8'), pretrained_model=None)
Runtime(name='med-3d-configurable-debug', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/med_3d_runner:v0.0.2', num_nodes=1, device='gpu

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [4]:
import os
import kubeflow.trainer

def list_files(mount_path: str):
    print(os.listdir(mount_path))

In [5]:
from kubeflow.trainer.options.kubernetes import RuntimePatch

pod_name = "node"
volume_name = "scratch-volume"
mount_path = "/scratch-volume"
volumes = [{"name": volume_name, "persistentVolumeClaim": {"claimName": volume_name}}]
volume_mounts = [{"name": volume_name, "mountPath": mount_path}]

volume_patch = RuntimePatch(
    training_runtime_spec={
        "template": {
            "spec": {
                "replicatedJobs": [
                    {
                        "name": "node",
                        "template": {
                            "spec": {
                                "template": {
                                    "spec": {
                                        "volumes": volumes,
                                        "containers": [
                                            {
                                                "name": pod_name,
                                                "volumeMounts": volume_mounts,
                                            }
                                        ],
                                    }   
                                }    
                            }
                        },
                    }
                ]
            }
        }
    }
)

In [6]:
from kubeflow.trainer import TrainerClient, CustomTrainer, options
from kubeflow.trainer.options import TrainerCommand
from kubeflow.trainer.types.types import CustomTrainerContainer

# ------------------------------------------------
# Configuration of resources with default notebeook namespace (2CPU;4GiMEM)
# ------------------------------------------------

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 1

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "4",           # CPUs per node
    "memory": "64Gi",     # Memory in GiB per node (tried 2Gi CrashLoopBackOff/OOMKilled), 64Gi works
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}

ENV_VARS = {
    "MASTER_ADDR": "localhost",
    "MASTER_PORT": "12355",
    "WORLD_SIZE": "1",
    "RANK": "0",
    "LOCAL_RANK": "0",
}

### AVAILABLE CONTAINER IMAGES

##displaying only index in the val dataset and use  # docker.io/pytorch/pytorch:2.9.1-cuda12.8-cudnn9-devel
## without ENV vars in dockerfile
##TODO run this with NUM_NODES = 2/"cpu": "8"/"memory": "264Gi"
# GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.1.4"

##displaying only index in the val dataset and use  # docker.io/pytorch/pytorch:2.9.1-cuda12.8-cudnn9-devel
## with ENV PYTHONUNBUFFERED=1 in dockerfile
# GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.1.3"
#works flooding cell with Waiting... (4000s)

##displaying only index in the val dataset and use  # docker.io/pytorch/pytorch:2.9.1-cuda12.8-cudnn9-devel
## with ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1 in dockerfile
#GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.1.2"
#works flooding cell with Waiting... (4000s)

# ## removed ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1 in dockerfile
GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.1.1"
# # it shows inconsistent logs but might related to the defined number of nodes
# # sometimes it keeps showing Waiting... (552s)
# # other times just showing Waiting... (34s) until 


command = TrainerCommand(
    command=[
        "torchrun",
        f"--nnodes={NUM_NODES}",
        "train_edm2.py", #path of script in scratch 
        "--outdir /scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY", # pragma: allowlist secret
        "--data /scratch-volume/data-fetal-us-edm2/FETAL_PLANES_DB",
        "--fpus23 /scratch-volume/data-fetal-us-edm2/FPUS23",
        "--african /scratch-volume/data-fetal-us-edm2/AfricanDataset/Zenodo_dataset",
        "--fetal-abdomen /scratch-volume/data-fetal-us-edm2/FetalAbdominalSegmentation/IMAGES",
        "--batch 4",
        "--preset edm2-img512-s",
        "--batch-gpu 4",
    ]
)

#NOTES
## torchrun commands for distributed learning but for this notebook resource are injected by Kubeflow trainer
        # "--standalone", #Single-node multi-worker # pragma: allowlist secret
        # f"--nnodes={NUM_NODES}",
        # f"--nproc_per_node={NPROC_PER_NODE}",#set at by Kubeflow trainer level
        # f"--node_rank={NODE_RANK}", #injected by Kubeflow trainer
        # f"--master_addr={MASTER_ADDR}",#injected by Kubeflow trainer
        # f"--master_port={MASTER_PORT}",#injected by Kubeflow trainer

## the following settings show some history of what has been tested
#(1/n: works)--------------------------------------
        # "--batch", "1",
        # "--preset", "edm2-img512-xxs",
        # "--batch-gpu", "1",

#(2/n: timeout)--------------------------------------
        # --batch 8 \
        # --preset edm2-img512-s \
        # --batch-gpu=8
### createse External termination (SIGTERM)  Most likely: timeout limit


In [7]:
job_id = trainer.train(
    runtime=trainer.get_runtime("torch-distributed"),
    trainer=CustomTrainerContainer(
        image=GITHUB_CONTAINER_REGISTRY,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
        env=ENV_VARS        
    ),
    options=[volume_patch, command],
)


In [8]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: d9a27a12fdd6
Job Status: Created
Creation Time: 2026-08-27 23:15:18+00:00

Job details: TrainJob(name='d9a27a12fdd6', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/ucl-arc-unified-ai/kubeflow-trainer-images-torch:v1.0.1', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=1, creation_timestamp=datetime.datetime(2026, 8, 27, 23, 15, 18, tzinfo=TzInfo(0)), status='Created')


In [ ]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


### EXPECTED TERMINAL LOG FROM LOCAL MACHINE:
#### COMMAND
# torchrun --standalone --nproc_per_node=1 train_edm2.py 
# --outdir ~/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY 
# --data ~/scratch-volume/FETAL_PLANES_DB             
# --batch 1         
# --preset edm2-img512-xxs
# --batch-gpu=1

#### LOGS
# Output directory:        /home/mxochicale/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY
# Dataset path:            /home/mxochicale/scratch-volume/FETAL_PLANES_DB
# Class-conditional:       True
# Number of GPUs:          1
# Batch size:              8
# Mixed-precision:         True

# Loading dataset...
# Dataset size: 7129
# Setting up encoder...
# Constructing network...
# {'class_name': 'training.networks_edm2.Precond', 'model_channels': 192, 'dropout': 0.0, 'use_fp16': True} {'img_resolution': 64, 'img_channels': 4, 'label_dim': 6}

# Precond                Parameters  Buffers  Output shape      Datatype
# ---                    ---         ---      ---               ---     
# unet.emb_fourier       -           384      [8, 192]          float32 
# unet.emb_noise         147456      -        [8, 768]          float32 
# unet.emb_label         4608        -        [8, 768]          float32 
# ...
# unet.dec.64x64_block3  1216513     -        [8, 192, 64, 64]  float16 
# unet.out_conv          6912        -        [8, 4, 64, 64]    float16 
# unet                   1           -        [8, 4, 64, 64]    float16 
# <top-level>            128         256      [8, 4, 64, 64]    float32 
# ---                    ---         ---      ---               ---     
# Total                  279449445   640      -                 -       
#
# Setting up training state...
# Training from 0 kimg to 2147483 kimg
#
# Setting up validation set...
# val dataset size: 5271
#   Processing 5271/5271 ... Pre-encoding validation latents... 
# Validation: 5271 images | sigma [0.019, 28.501]
#   0%|          | 0/131072 [00:00<?, ?it/s]NETWORK: Precond(
#   (unet): UNet(
#     (emb_fourier): MPFourier()
#     (emb_noise): MPConv()
#     (emb_label): MPConv()
#     (enc): ModuleDict(
#       (64x64_conv): MPConv()
#       (64x64_block0): Block(
#         (conv_res0): MPConv()
#         (emb_linear): MPConv()
#         (conv_res1): MPConv()
#       )
#     ...
#       (64x64_block3): Block(
#         (conv_res0): MPConv()
#         (emb_linear): MPConv()
#         (conv_res1): MPConv()
#         (conv_skip): MPConv()
#       )
#     )
#     (out_conv): MPConv()
#   )
#   (logvar_fourier): MPFourier()
#   (logvar_linear): MPConv()
# )
# --------------------
#   0%|          | 0/131072 [00:00<?, ?it/s]Status: kimg 0.0       time 30m 42s      
#                                                                  sec/tick 1841.65  sec/kimg 0.000   maintenance 1841.65 cpumem 4.18   gpumem 2.09   reserved 2.86  
# Val_loss:  1.5974
# Saving network-snapshot-0000000-0.050.pkl ... done
# Saving network-snapshot-0000000-0.100.pkl ... done
#  18%|#8        | 23843/131072 [2:05:01<9:22:16,  3.18it/s]    
# ...
#

Waiting for job logs...
[23:15:21] Waiting... (1s)
[23:15:22] Waiting... (2s)
[23:15:23] Waiting... (3s)
[23:15:24] Waiting... (4s)
[23:15:25] Waiting... (5s)
[23:15:26] Waiting... (6s)
[23:15:27] Waiting... (7s)
[23:15:28] Waiting... (8s)
[23:15:30] Waiting... (9s)
[23:15:31] Waiting... (10s)
[23:15:32] Waiting... (11s)
[23:15:33] Waiting... (12s)
[23:15:34] Waiting... (13s)
[23:15:35] Waiting... (14s)
[23:15:36] Waiting... (15s)
[23:15:37] Waiting... (16s)
[23:15:38] Waiting... (17s)
[23:15:39] Waiting... (18s)
[23:15:40] Waiting... (19s)
[23:15:41] Waiting... (20s)
[23:15:42] Waiting... (21s)
[23:15:43] Waiting... (22s)
[23:15:44] Waiting... (23s)
[23:15:45] Waiting... (24s)
[23:15:46] Waiting... (25s)
[23:15:47] Waiting... (26s)
